# 🧠 TRACE — Colab Brain

This notebook is the **AI thinking layer** for the TRACE/Friday assistant.
It connects to your local machine over a WebSocket tunnel (ngrok), receives
tool schemas, runs the Gemini tool-calling loop, and delegates all actual
tool execution back to your local ToolRouter.

### Setup steps
1. Run **Cell 1** to install dependencies
2. Fill in **Cell 2** with your secrets and ngrok URL
3. Run **Cell 3** to start the brain loop

### How to stop
Interrupt the kernel (■). Your local Friday server will detect the
disconnect and automatically fall back to its built-in local agent.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run once per Colab session
!pip install -q google-genai websockets

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
# Fill in these values before running Cell 3.

# Your local Friday server exposed via ngrok (or Cloudflare Tunnel).
# Example: 'wss://abc123.ngrok-free.app/ws/colab'
# If running locally without a tunnel: 'ws://localhost:8000/ws/colab'
LOCAL_WS_URL = "wss://YOUR_NGROK_URL_HERE/ws/colab"

# Must match COLAB_SECRET in your local .env
COLAB_SECRET = "colab-change-me"

# Your Gemini API key
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"

# Gemini model to use for thinking (can differ from local model)
# For heavy reasoning: 'gemini-2.0-flash-thinking-exp' or 'gemini-2.5-pro'
# For speed: 'gemini-2.5-flash'
GEMINI_MODEL = "gemini-2.5-flash"

print(f"Config loaded. Model: {GEMINI_MODEL}")
print(f"Will connect to: {LOCAL_WS_URL}")

In [ ]:
# ── Cell 3: Brain Loop ────────────────────────────────────────────────────────
# This cell blocks and runs the brain. Interrupt kernel to stop.

import asyncio
import json
import logging
import uuid

import websockets
from google import genai
from google.genai import types

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("colab_brain")


# ── Schema helpers (mirrors agent.py / model_flow.py) ────────────────────────

def _clean_schema(schema: dict, for_gemini: bool = True) -> dict:
    """Strip unsupported keys; uppercase type values for Gemini."""
    if not isinstance(schema, dict):
        return schema
    allowed = {"type", "format", "description", "enum", "properties", "required", "items"}
    cleaned = {}
    for k, v in schema.items():
        if k not in allowed:
            continue
        if k == "type" and isinstance(v, str):
            cleaned[k] = v.upper() if for_gemini else v
        elif k == "properties":
            cleaned[k] = {pk: _clean_schema(pv, for_gemini) for pk, pv in v.items()}
        elif k == "items":
            cleaned[k] = _clean_schema(v, for_gemini)
        else:
            cleaned[k] = v
    return cleaned


def _to_gemini_tool(tool_dict: dict) -> dict:
    raw = tool_dict.get("input_schema", {})
    if raw.get("type") in ("object", "OBJECT"):
        params = _clean_schema(raw)
    else:
        params = _clean_schema({"type": "OBJECT", "properties": raw})
    return {
        "name": tool_dict["name"],
        "description": tool_dict["description"],
        "parameters": params,
    }


# ── Brain core ────────────────────────────────────────────────────────────────

SYSTEM_INSTRUCTION = (
    "You are Friday, a helpful desktop AI assistant running on the user's Linux machine. "
    "You have access to tools — use them proactively.\n\n"
    "TERMINAL (create_terminal_session, write_to_terminal, read_from_terminal):\n"
    "- You CAN run shell commands. For system tasks (updates, file ops, running scripts), "
    "create a terminal session and execute the command. Always read output after writing.\n\n"
    "WEB FETCH (web_fetch):\n"
    "- To read any URL, always use web_fetch first. Works without a browser extension.\n\n"
    "BROWSER TOOLS (browser_*):\n"
    "- Only work when the Friday Chrome extension is connected. "
    "If a browser tool returns 'extension is not connected', fall back to web_fetch.\n\n"
    "TASKS (create_task, get_tasks, update_task, delete_task):\n"
    "- Manage the user's to-do list. Do NOT confuse 'update my system' with update_task.\n\n"
    "RULES:\n"
    "- Never refuse if you have a capable tool. Ask for confirmation before destructive commands."
)


async def run_turn(
    ws,
    turn_id: str,
    message: str,
    history: list[dict],
    gemini_tools: list[dict],
    client: genai.Client,
) -> None:
    """Run a full Gemini tool-calling loop for one user turn."""

    # Rebuild history in Gemini format
    gemini_history = []
    for turn in history:
        role = turn.get("role", "user")
        content = turn.get("content", "")
        gemini_history.append(
            types.Content(role=role, parts=[types.Part(text=content)])
        )

    chat = client.chats.create(
        model=GEMINI_MODEL,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            tools=[{"function_declarations": gemini_tools}],
            temperature=0.7,
        ),
        history=gemini_history,
    )

    try:
        response = chat.send_message(message)

        # Tool-calling loop
        while response.function_calls:
            tool_parts = []

            for fc in response.function_calls:
                name = fc.name
                args = {k: v for k, v in fc.args.items()}

                logger.info("[Turn %s] Tool call: %s(%s)", turn_id[:8], name, args)

                # Send tool_call to local machine
                await ws.send(json.dumps({
                    "type": "tool_call",
                    "id": turn_id,
                    "name": name,
                    "args": args,
                }))

                # Wait for tool_result from local machine
                result_msg = json.loads(await ws.recv())
                result_str = result_msg.get("result", "")
                logger.info("[Turn %s] Tool result: %s…", turn_id[:8], result_str[:120])

                tool_parts.append(
                    types.Part.from_function_response(
                        name=name,
                        response={"result": result_str},
                    )
                )

            response = chat.send_message(tool_parts)

        # Send final text
        text = response.text or ""
        if text:
            await ws.send(json.dumps({
                "type": "text_chunk",
                "id": turn_id,
                "content": text,
            }))

    except Exception as exc:
        logger.exception("[Turn %s] Gemini error: %s", turn_id[:8], exc)
        await ws.send(json.dumps({
            "type": "error",
            "id": turn_id,
            "content": str(exc),
        }))

    finally:
        await ws.send(json.dumps({"type": "done", "id": turn_id}))
        logger.info("[Turn %s] Done.", turn_id[:8])


async def brain_loop():
    """Main loop: connect, receive schemas, handle turns forever."""
    client = genai.Client(api_key=GEMINI_API_KEY)
    url = f"{LOCAL_WS_URL}?secret={COLAB_SECRET}"

    print(f"\n🧠 Connecting to Friday at {LOCAL_WS_URL} …")

    async with websockets.connect(url, ping_interval=30, ping_timeout=10) as ws:
        print("✅ Connected! Waiting for tool schemas …")

        # First message must be tool_schemas
        schemas_msg = json.loads(await ws.recv())
        assert schemas_msg["type"] == "tool_schemas", f"Expected tool_schemas, got {schemas_msg}"

        tool_list = schemas_msg["tools"]
        gemini_tools = [_to_gemini_tool(t) for t in tool_list]
        print(f"📦 Received {len(gemini_tools)} tool schemas:")
        for t in tool_list:
            print(f"   • {t['name']}")

        print("\n🟢 Brain is live. Waiting for user messages …\n")

        # Main receive loop
        async for raw in ws:
            msg = json.loads(raw)
            msg_type = msg.get("type")

            if msg_type == "user_message":
                turn_id = msg["id"]
                message = msg["message"]
                history = msg.get("history", [])
                logger.info("[Turn %s] User: %s", turn_id[:8], message[:80])

                # Process each turn; tool_call/tool_result messages during
                # run_turn are handled inline (single-threaded per turn)
                await run_turn(ws, turn_id, message, history, gemini_tools, client)

            elif msg_type == "tool_schemas":
                # Server re-sent schemas (e.g. after tool registration change)
                tool_list = msg["tools"]
                gemini_tools = [_to_gemini_tool(t) for t in tool_list]
                print(f"🔄 Tool schemas refreshed ({len(gemini_tools)} tools)")

            else:
                logger.debug("Unexpected message type: %s", msg_type)


# ── Run ───────────────────────────────────────────────────────────────────────
print("Starting Colab Brain …")
try:
    asyncio.run(brain_loop())
except KeyboardInterrupt:
    print("\n⛔ Brain stopped. Local Friday will use its built-in agent as fallback.")

## 🔗 Tunnel Setup (run on your local machine)

The Colab notebook needs a public URL to reach your `localhost:8000`.

### Option A — ngrok (free, URL changes each session)
```bash
# Install
pip install pyngrok

# Start tunnel (in a new terminal, with venv active)
ngrok http 8000
# → Copy the 'Forwarding' URL, e.g. https://abc123.ngrok-free.app
# → Set LOCAL_WS_URL = 'wss://abc123.ngrok-free.app/ws/colab'
```

### Option B — Cloudflare Tunnel (free, stable URL tied to your domain)
```bash
# Install cloudflared, then:
cloudflared tunnel --url http://localhost:8000
```

### Your .env settings
```env
COLAB_SECRET=your-strong-secret-here
COLAB_MODE=true
```

Set the same `COLAB_SECRET` in Cell 2 above.

## 🗑️ How to Remove the Colab Feature Entirely

1. Delete `backend/bridge/` directory
2. Delete `backend/routes/colab_ws.py`
3. In `backend/main.py` — remove the 3 lines marked `# ← COLAB BRIDGE`
4. In `backend/routes/chat.py` — remove the 2 blocks between `── COLAB BRIDGE` and `── END COLAB BRIDGE`
5. In `backend/config.py` — remove the `colab_secret` and `colab_mode` lines
6. Delete this notebook (`colab_brain.ipynb`)

Everything else remains untouched.